In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


In [ ]:
from google.colab import userdata
from huggingface_hub import login
llama_token = userdata.get('llama_token')
login(llama_token)

In [ ]:
# Настройка промпта для Bank
prompt_config = {
        "task": "Predict whether a bank client will subscribe",
        "pos_label": "yes",
        "neg_label": "no",
        "entity": "subscription",
        "question": "Will this client subscribe?"
}

FEATURE_MAPPINGS = {
        'V1': 'Age',
        'V2': 'Job',
        'V3': 'Martial',
        'V4': 'Education',
        'V5': 'Default',
        'V6': 'Balance',
        'V7': 'Housing',
        'V8': 'Loan',
        'V9': 'Contact',
        'V10': 'Day of Week',
        'V11': 'Month',
        'V12': 'Duration',
        'V13': 'Campaign',
        'V14': 'Pdays',
        'V15': 'Previous',
        'V16': 'Poutcome',
        'Class': 'y'
}

openml_id = 1461

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    df = df.rename(columns=FEATURE_MAPPINGS)
    feature_names = list(FEATURE_MAPPINGS.values())[:-1]
    target_name = list(FEATURE_MAPPINGS.values())[-1]
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name

def split_dataset(df, target_name, test_size=0.2, seed=42):

    # Разделение на train/test (80/20)
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    return train_df, test_df

df, feature_names, target_name = load_dataset(openml_id)
train_df, test_df = split_dataset(df, target_name)

In [ ]:
df[target_name].value_counts()

,count
y,
0,39922
1,5289


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   Age          45211 non-null  int64   
 1   Job          45211 non-null  category
 2   Martial      45211 non-null  category
 3   Education    45211 non-null  category
 4   Default      45211 non-null  category
 5   Balance      45211 non-null  int64   
 6   Housing      45211 non-null  category
 7   Loan         45211 non-null  category
 8   Contact      45211 non-null  category
 9   Day of Week  45211 non-null  int64   
 10  Month        45211 non-null  category
 11  Duration     45211 non-null  int64   
 12  Campaign     45211 non-null  int64   
 13  Pdays        45211 non-null  int64   
 14  Previous     45211 non-null  int64   
 15  Poutcome     45211 non-null  category
 16  y            45211 non-null  int64   
dtypes: category(9), int64(8)
memory usage: 3.1 MB


In [ ]:
for name, df in [("train", train_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")


train (всего: 36168):
  no: 31937 — 88.3%
  yes:  4231 — 11.7%

test (всего: 9043):
  no:  7985 — 88.3%
  yes:  1058 — 11.7%


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names, target_name, prompt_config=None, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    if include_target and prompt_config is not None:
        target_value = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']
        text += f": {target_name} -> {target_value}"

    return text

# Тест
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, True))
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, False))
train_df.head(1)

The value of Age is 36. The category of Job is technician. The category of Martial is divorced. The category of Education is secondary. The category of Default is no. The value of Balance is 861. The category of Housing is no. The category of Loan is no. The category of Contact is telephone. The value of Day of Week is 29. The category of Month is aug. The value of Duration is 140. The value of Campaign is 2. The value of Pdays is -1. The value of Previous is 0. The category of Poutcome is unknown.: y -> no
The value of Age is 36. The category of Job is technician. The category of Martial is divorced. The category of Education is secondary. The category of Default is no. The value of Balance is 861. The category of Housing is no. The category of Loan is no. The category of Contact is telephone. The value of Day of Week is 29. The category of Month is aug. The value of Duration is 140. The value of Campaign is 2. The value of Pdays is -1. The value of Previous is 0. The category of Pout

,Age,Job,Martial,Education,Default,Balance,Housing,Loan,Contact,Day of Week,Month,Duration,Campaign,Pdays,Previous,Poutcome,y
24001,36,technician,divorced,secondary,no,861,no,no,telephone,29,aug,140,2,-1,0,unknown,0


In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    pos = prompt_config['pos_label'].lower()
    neg = prompt_config['neg_label'].lower()

    # Точное совпадение
    if response == pos:
        return 1
    elif response == neg:
        return 0

    # Начинается с метки
    if response.startswith(pos):
        return 1
    elif response.startswith(neg):
        return 0

    # Содержит метку как отдельное слово
    words = response.split()
    if pos in words:
        return 1
    elif neg in words:
        return 0

    # Не удалось распознать
    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'yes'
Prediction: 1

Response: 'hi'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    pr  = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return roc, f1, acc, pr, rec


# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name, prompt_config)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The value of Age is 40. The category of Job is blue-collar. The category of Martial is married. The category of Education is primary. The category of Default is no. The value of Balance is 640. The category of Housing is yes. The category of Loan is yes. The category of Contact is unknown. The value


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name, prompt_config)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The value of Age is 40. The category of Job is blue-collar. The category of Martial is married. The category of Education is primary. The category of Default is no. The value of Balance is 640. The category of Housing is yes. The category of Loan is yes. The category of Contact is unknown. The value


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Название модели: Llama 3.1 8B Instruct от Meta
model_name = "meta-llama/Llama-3.1-8B-Instruct"

# Определение устройства: приоритет отдается GPU с поддержкой CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

# Загрузка токенизатора для Llama 3.1
# Llama 3 использует расширенный словарь (128k токенов)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Установка токена заполнения (pad_token)
# В моделях Llama часто отсутствует pad_token, поэтому используем eos_token
tokenizer.pad_token = tokenizer.eos_token

# Загрузка базовой модели с использованием bfloat16 для оптимизации памяти и стабильности
# Параметр device_map="auto" автоматически распределяет слои модели по доступным ресурсам
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="sdpa"
)

# Проверка выделенной видеопамяти после загрузки модели
if torch.cuda.is_available():
    allocated_memory = torch.cuda.memory_allocated() / 1e9
    print(f"Занято видеопамяти на GPU: {allocated_memory:.2f} GB")

Используемое устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Занято видеопамяти на GPU: 16.06 GB


In [ ]:
def create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"{prompt_config['entity']} information: "
            f"{row_to_text_template(row, feature_names, target_name, prompt_config)}\n"
            f"{prompt_config['question']}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name, prompt_config)
            ex_target = prompt_config['pos_label'] if ex[target_name] == 1 else prompt_config['neg_label']

            messages.append({
                "role": "user",
                "content": f"{prompt_config['entity']} information: {ex_text}\n{prompt_config['question']}"
            })
            messages.append({
                "role": "assistant",
                "content": ex_target
            })

        client_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        messages.append({
            "role": "user",
            "content": f"{prompt_config['entity']} information: {client_text}\n{prompt_config['question']}"
        })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для меток из prompt_config
    first_token_logits = outputs.scores[0][0]

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]

    pos_logit = first_token_logits[pos_id]
    neg_logit = first_token_logits[neg_id]

    probs = F.softmax(torch.stack([pos_logit, neg_logit]), dim=0)
    prob_pos = probs[0].item()

    return response, prob_pos

# Тест
test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, prob = predict_single_with_prob(test_prompt, prompt_config, model, tokenizer, device)
print(f"Response: '{response}', Probability: {prob:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: 'no', Probability: 0.1824


# 3.1 Zero-shot

In [ ]:
print("Zero-shot классификация")

seed = 42

# Запускаем zero-shot предсказания
y_true_zero = []
y_pred_zero = []
y_prob_zero = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
    response, prob = predict_single_with_prob(prompt, prompt_config, model, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_zero.append(row[target_name])
    y_pred_zero.append(prediction)
    y_prob_zero.append(prob)

zero_shot_time = time.time() - start_time

print(f"zero_shot_time: {zero_shot_time}")
print(f"zero_shot_time / len(y_true_zero) {zero_shot_time / len(y_true_zero)}")

Zero-shot классификация


  0%|          | 0/9043 [00:00<?, ?it/s]

zero_shot_time: 694.1888811588287
zero_shot_time / len(y_true_zero) 0.07676533021771854


In [ ]:
# Вычисляем метрики
roc_zero, f1_zero, acc_zero, pr_zero, rec_zero = compute_metrics(np.array(y_true_zero), np.array(y_pred_zero), np.array(y_prob_zero))

print("Результаты zero-shot:")
print(f"ROC AUC: {roc_zero}")
print(f"F1 Score: {f1_zero}")
print(f"Accuracy: {acc_zero}")
print(f"Precision: {pr_zero}")
print(f"Recall: {rec_zero}")

Результаты zero-shot:
ROC AUC: 0.5619980398028914
F1 Score: 0.0
Accuracy: 0.8830034280659074
Precision: 0.0
Recall: 0.0


In [ ]:
zero_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_zero),
    np.array(y_pred_zero),
    np.array(y_prob_zero),
    n_iter=1000
)

print("\nРезультаты zero-shot (bootstrap метрики с доверительными интервалами):")
for key, value in zero_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты zero-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.5621±0.0105
  F1: 0.0000±0.0000
  Accuracy: 0.8829±0.0034
  Precision: 0.0000±0.0000
  Recall: 0.0000±0.0000


Время тестирования: 12 мин.

# 3.2 Few-shot

In [ ]:
# Выбираем 64 примера из train для контекста
n_examples = 64

seed = 42

# Балансированная выборка
n_per_class = n_examples // 2
pos_examples = train_df[train_df[target_name] == 1].sample(n=n_per_class, random_state=seed)
neg_examples = train_df[train_df[target_name] == 0].sample(n=n_per_class, random_state=seed)
few_shot_df = pd.concat([pos_examples, neg_examples]).sample(frac=1, random_state=seed)

print(f"\nВыбрано {len(few_shot_df)} примеров для контекста:")
print(f"  {prompt_config['pos_label']}: {(few_shot_df[target_name] == 1).sum()}")
print(f"  {prompt_config['neg_label']}: {(few_shot_df[target_name] == 0).sum()}")

few_shot_examples = [row for _, row in few_shot_df.iterrows()]

# Few-shot предсказания
y_true_few = []
y_pred_few = []
y_prob_few = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples)
    response, prob = predict_single_with_prob(prompt, prompt_config, model, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_few.append(row[target_name])
    y_pred_few.append(prediction)
    y_prob_few.append(prob)

few_shot_time = time.time() - start_time

print(f"few_shot_time: {few_shot_time}")
print(f"few_shot_time / len(y_true_few) {few_shot_time / len(y_true_few)}")


Выбрано 64 примеров для контекста:
  yes: 32
  no: 32


  0%|          | 0/9043 [00:00<?, ?it/s]

few_shot_time: 7741.197350502014
few_shot_time / len(y_true_few) 0.8560430554574825


In [ ]:
roc_few, f1_few, acc_few, pr_few, rec_few = compute_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few)
)
print("Результаты few-shot:")
print(f"ROC AUC: {roc_few}")
print(f"F1 Score: {f1_few}")
print(f"Accuracy: {acc_few}")
print(f"Precision: {pr_few}")
print(f"Recall: {rec_few}")


Результаты few-shot:
ROC AUC: 0.6425101176236635
F1 Score: 0.2248388696083292
Accuracy: 0.30841534888864314
Precision: 0.12938659058487875
Recall: 0.8572778827977315


In [ ]:
few_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few),
    n_iter=1000
)

print("\nРезультаты few-shot (bootstrap метрики с доверительными интервалами):")
for key, value in few_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты few-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.6420±0.0093
  F1: 0.2250±0.0062
  Accuracy: 0.3084±0.0049
  Precision: 0.1295±0.0040
  Recall: 0.8569±0.0105


In [ ]:
from google.colab import runtime
runtime.unassign()

Время выполнения: ~ 2 ч 10 мин.

Использование оперативная памяти графического процесса: 18.2/80GB.

Графический процессор A100.